# Exercise: Extend and Evaluate a LangChain RAG Pipeline

In the lesson you built a RAG pipeline with LangChain. You start from the working pipeline and take it further: make its answers **traceable** by returning their sources, **tune** how much context it retrieves, and then **evaluate** the result so you can say which setup is better, not just guess.

For the concepts behind evaluation, see [RAG Concepts](1_rag_concepts.md).

## Learning Objectives

At the end of this notebook, you should be able to:

- Extend a RAG pipeline to return the source passages behind each answer.
- Tune retrieval (the number of chunks) and observe its effect on the answer.
- Evaluate and compare two configurations with an LLM-as-judge (faithfulness and answer relevance).

## Provided: the pipeline from the lesson

Run this cell as is. It gives you `build_chain(pdf_path, k=..., chunk_size=..., chunk_overlap=...)`, which returns a ready retrieval chain, and `ask(question, chain)`, which invokes a chain and returns the full result (the `input`, the retrieved `context`, and the `answer`). It also builds `react_chain` over the React paper for you. You do not need to change anything here.

In [1]:
import warnings

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client
from pypdf import PdfReader

/home/hashim/all_imp_repos/ds-rag-pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
warnings.filterwarnings("ignore")

In [3]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [4]:
def build_chain(pdf_path, k=4, chunk_size=200, chunk_overlap=50):
    """Load a PDF, build a Chroma retriever, and wire it to the LLM. Returns a retrieval chain."""
    reader = PdfReader(pdf_path)

    docs = [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": pdf_path, "page": i},
        )
        for i, page in enumerate(reader.pages, start=1)
    ]

    chunks = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    ).split_documents(docs)

    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        encode_kwargs={"normalize_embeddings": True},
    )

    store = Chroma.from_documents(
        chunks, embedding, collection_metadata={"hnsw:space": "cosine"}
    )

    retriever = store.as_retriever(search_kwargs={"k": k})

    prompt = Client().pull_prompt(
        "langchain-ai/retrieval-qa-chat",
        include_model=False,
        dangerously_pull_public_prompt=True,
    )

    combine = create_stuff_documents_chain(llm=llm, prompt=prompt)
    return create_retrieval_chain(retriever=retriever, combine_docs_chain=combine)


def ask(question, chain):
    """Invoke a chain and return the full result dict (input, context, answer)."""
    return chain.invoke({"input": question})


# Build the pipeline over the React paper, ready to use
react_chain = build_chain("documents/react_paper.pdf")

## Exercise 1: Make answers traceable with source citations

A RAG answer is only trustworthy if you can check where it came from. The result from `ask` includes a `context` list of the retrieved documents, and each document carries metadata, including its page number in `doc.metadata`.

Write a function that asks a question and returns the answer **together with the source pages** of the chunks that supported it.

Hints:

- Call `ask(question, react_chain)` and read `result["answer"]` and `result["context"]`.
- Each item in `context` is a `Document`; look at `doc.metadata` to find the page.

In [5]:
def answer_with_sources(question):
    """Return the answer plus the source pages of the retrieved chunks."""
    result = ask(question, react_chain)
    pages = sorted({doc.metadata["page"] for doc in result["context"]})
    return result["answer"], pages

In [6]:
answer, pages = answer_with_sources("How does ReAct perform on HotpotQA compared to chain-of-thought?")
print(answer)
print(f"\nSources: pages {pages}")

On HotpotQA, ReAct is a bit weaker than chain‑of‑thought.  In the reported experiments ReAct scored **27.4 %** while CoT scored **29.4 %**, so ReAct lags behind CoT by roughly two percentage points.

Sources: pages [5, 6, 14]


## Exercise 2: Tune how much context you retrieve

The retriever returns the top `k` chunks. Too few may miss the answer, too many may dilute the prompt with irrelevant text. `build_chain` accepts a `k` argument, so you can build chains with different values.

Build two chains over the React paper, one with a small `k` and one with a larger `k`, ask both the same question, and compare the answers.

Hint: `chain_small = build_chain("documents/react_paper.pdf", k=2)`.

In [7]:
QUESTION = "What are the main limitations of ReAct reported in the paper?"

chain_small = build_chain("documents/react_paper.pdf", k=2)
chain_large = build_chain("documents/react_paper.pdf", k=6)

result_small = ask(QUESTION, chain_small)
result_large = ask(QUESTION, chain_large)

print("=== k=2 ===\n" + result_small["answer"])
print("\n=== k=6 ===\n" + result_large["answer"])

=== k=2 ===
The excerpt you provided only discusses how a single hallucinating sentence in Act 17 and a few hints in Act 23 can alter ReAct’s behavior. It does not mention or describe any limitations of ReAct. Therefore, based solely on the given context, I can’t identify the main limitations reported in the paper.

=== k=6 ===
The paper highlights a few key limitations of the ReAct framework:

1. **Hallucinations in the reasoning trace** – The authors show that a single hallucinating sentence (e.g., in Act 17) can derail the entire chain of reasoning. Even a small hallucination can lead ReAct to produce incorrect or misleading conclusions.

2. **Sensitivity to the exact wording of the trace** – Small edits to the trace (such as removing a hallucinated sentence or adding a hint in Act 23) can dramatically change the model’s behavior. This indicates that ReAct is fragile and highly dependent on the precise phrasing of the reasoning steps.

3. **Greedy decoding as a bottleneck** – The au

## Exercise 3: Evaluate and compare the two configurations

Comparing answers by eye does not scale. Use the model as a judge to score each answer, then let the scores decide which `k` worked better.

Write a small `llm_judge` function that asks the model to rate something from 1 to 5, then use it to score two things for each answer:

- **Faithfulness**: is the answer supported by the retrieved context?
- **Answer relevance**: does the answer address the question?

Compare the scores for your small-`k` and large-`k` chains, and write a sentence on which you would choose and why.

Hint: build the judging prompt from the question, the context, and the answer, send it with the provided `llm`, and parse a single digit from the reply.

In [8]:
import re


def llm_judge(instruction):
    """Ask the model to rate something from 1 to 5 and return the number."""
    reply = llm.invoke(
        instruction + "\n\nReply with a single integer from 1 to 5 and nothing else."
    ).content
    match = re.search(r"[1-5]", reply)
    return int(match.group()) if match else None

In [9]:
def evaluate(result):
    """Score one chain result for faithfulness and answer relevance."""
    context = "\n---\n".join(doc.page_content for doc in result["context"])
    faithfulness = llm_judge(
        "Rate how well the ANSWER is supported by the CONTEXT "
        "(5 = every claim is supported, 1 = mostly unsupported).\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {result['input']}\n\nANSWER: {result['answer']}"
    )
    relevance = llm_judge(
        "Rate how well the ANSWER addresses the QUESTION "
        "(5 = fully answers it, 1 = does not address it).\n\n"
        f"QUESTION: {result['input']}\n\nANSWER: {result['answer']}"
    )
    return {"faithfulness": faithfulness, "relevance": relevance}


for name, result in [("k=2", result_small), ("k=6", result_large)]:
    print(name, evaluate(result))

k=2 {'faithfulness': 5, 'relevance': 1}


k=6 {'faithfulness': 5, 'relevance': 5}


**Which `k` to choose:** `k=6`. Both answers scored 5 for faithfulness, because neither claimed
anything outside its context. But `k=2` scored 1 for relevance: its two chunks did not cover the
limitations discussion, so it could only say it did not know. `k=6` pulled in enough of the
paper's discussion to answer fully and scored 5 for relevance. With chunks of only 200 characters,
`k=2` is just about 400 characters of context, which is too little for a question whose answer
is spread across a section.

## Conclusion and Reflection

- Did citations change how much you trust the answers? When would source pages matter most?
- Which `k` gave better judge scores, and did that match your own reading of the answers?
- The judge is itself an LLM and can be biased. How would you make the evaluation more reliable, and where would a tool like RAGAS or a labelled test set help?

**My answers**

- **Citations and trust:** the page numbers make an answer checkable. The HotpotQA numbers
  (ReAct 27.4 vs. CoT 29.4) can be checked directly on the cited pages. Sources matter most when
  answers feed decisions, such as medical, legal or financial ones, and when the corpus has many
  documents, so a reader needs to know which one an answer came from.
- **Judge vs. my own reading:** `k=6` scored better (5/5 vs. 5/1), which matches my reading. The
  `k=2` answer was honest but useless. It also shows why faithfulness alone is not enough: a
  refusal is perfectly "faithful".
- **More reliable evaluation:** use a different (ideally stronger) model as the judge than the
  generator, average several runs or several judges, use clear rubrics with examples, and above all
  evaluate over a labelled test set of many questions rather than one. RAGAS provides standard
  metrics for this (faithfulness, answer relevancy, context precision and recall), and the
  labelled set gives ground truth that the judge's scores can be checked against.